# 栅栏密码 (Rail Fence Cipher) — 自学课程

**分类**：古典密码学

**内容简介**：基于Z字形排列的换位密码，通过调整字母顺序加密，是经典教学案例



## 学习目标

- 理解算法规则与数学表达
- 实现加解密函数，并写基本测试
- 通过小实验观察安全性与攻击方法
- 能解释常见踩坑并独立调试



## 前置知识与符号约定

- 字母表：仅使用大写 A-Z
- 映射：$A\rightarrow 0,\dots,Z\rightarrow 25$
- 模运算：$x\bmod 26$ 结果落在 $\{0,\dots,25\}$

> 如果你希望支持中文/标点，本课程也会在后续练习中给出扩展思路。



In [ ]:
# 课程通用工具：字母映射与规范化
import string
from collections import Counter, defaultdict
import math, random

ALPHABET = string.ascii_uppercase
A2I = {ch:i for i,ch in enumerate(ALPHABET)}
I2A = {i:ch for i,ch in enumerate(ALPHABET)}

def normalize(text: str) -> str:
    """仅保留 A-Z 并转大写"""
    return ''.join(ch for ch in text.upper() if ch in ALPHABET)

def keep_nonletters_encrypt(text: str, enc_func):
    """对字母加密，非字母原样保留（用于扩展练习）"""
    out=[]
    for ch in text:
        if ch.upper() in ALPHABET:
            out.append(enc_func(ch.upper()))
        else:
            out.append(ch)
    return ''.join(out)

print(normalize("Hello, World! 123"))
# 预期输出：HELLOWORLD



# Step 1：把算法写成数学模型

我们用统一抽象描述密码：

- 加密：$E_k: \mathcal{P}\to\mathcal{C}$
- 解密：$D_k: \mathcal{C}\to\mathcal{P}$
- 正确性：

$$D_k(E_k(p))=p$$

这能帮助你：
- 写出可测试的实现
- 分析密钥空间大小
- 讨论攻击模型（已知明文、选择明文等）



## 自检小测

1) 什么是“模 26”运算？为什么它会让结果回到 0..25？
2) 为什么实现中必须统一 A→0 的映射？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



> 常见错误 / 踩坑点 / 调试提示：
>
> - 不要混用 A→1 与 A→0 两种映射。
> - 记得对 k 做 k%=26；否则 k>26 时结果可能不符合预期。
> - 先写 round-trip 测试：decrypt(encrypt(p,k),k)==p。



# Step 2：栅栏密码（置换）

栅栏密码把字符按轨道“之”字形排列，再按轨道读出。



## 自检小测

1) 为什么它不改变字母频率？这意味着什么？
2) rails=2 与 rails=3 在结构上有什么不同？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



# Step 2：栅栏密码原理

栅栏密码（Rail Fence）是典型的置换密码：

- 把明文沿着“之”字形写到 $r$ 条轨道
- 再按行读出得到密文

> 安全直觉：它不改变字母频率，因此容易被频率分析与结构性攻击结合破解。



In [ ]:
# Step 2：栅栏路径生成
def rail_pattern(n: int, rails: int):
    if rails < 2:
        raise ValueError("rails 至少为 2")
    r=0
    step=1
    for _ in range(n):
        yield r
        if r == 0:
            step = 1
        elif r == rails-1:
            step = -1
        r += step

print(list(rail_pattern(10, 3)))
# 预期输出：类似 [0,1,2,1,0,1,2,1,0,1]



In [ ]:
# Step 3：加密
from collections import defaultdict

def rail_encrypt(text: str, rails: int) -> str:
    t=normalize(text)
    buckets=defaultdict(list)
    for ch, r in zip(t, rail_pattern(len(t), rails)):
        buckets[r].append(ch)
    return ''.join(''.join(buckets[r]) for r in range(rails))

print(rail_encrypt("WEAREDISCOVEREDFLEEATONCE", 3))
# 预期输出：经典例子常见密文：WECRLTEERDSOEEFEAOCAIVDEN（可能因示例略有不同）



In [ ]:
# Step 4：解密（先还原每条轨道长度，再按路径回填）
def rail_decrypt(cipher: str, rails: int) -> str:
    c=normalize(cipher)
    n=len(c)
    pattern=list(rail_pattern(n, rails))
    counts=[pattern.count(r) for r in range(rails)]
    # 切片分配给各轨道
    rails_str=[]
    idx=0
    for r in range(rails):
        rails_str.append(list(c[idx:idx+counts[r]]))
        idx += counts[r]
    # 按 pattern 取回
    pointers=[0]*rails
    out=[]
    for r in pattern:
        out.append(rails_str[r][pointers[r]])
        pointers[r]+=1
    return ''.join(out)

ct=rail_encrypt("WEAREDISCOVEREDFLEEATONCE", 3)
print(ct)
print(rail_decrypt(ct, 3))
# 预期输出：第二行还原 WEAREDISCOVEREDFLEEATONCE



# Step 5：爆破 rails 参数

如果不知道 rails，可以对 $r=2..10$ 做爆破，结合英文评分筛选。



In [ ]:
# Step 5：爆破 rails
def crack_rail(cipher: str, max_rails: int = 10, topk: int = 5):
    c=normalize(cipher)
    cand=[]
    for r in range(2, max_rails+1):
        pt=rail_decrypt(c, r)
        cand.append((score_english_like(pt), r, pt))
    cand.sort(reverse=True)
    return cand[:topk]

for s,r,pt in crack_rail(ct, max_rails=8, topk=5):
    print(s, r, pt[:40])
# 预期输出：应包含 r=3 的正确明文候选



## 练习

1) 让算法支持保留非字母字符（与凯撒练习类似）。
2) 思考：栅栏密码为什么属于“置换密码”？它改变了什么，不改变什么？
3) 组合攻击：栅栏 + 凯撒（先凯撒后栅栏），你能爆破出两层参数吗？



# Step 6：复杂度与实用建议

- 加密/解密时间复杂度约为 $O(n)$
- rails 太大时，结构更接近“写成矩阵再读”，安全性仍有限

> 建议：把它当作理解置换的入门，而不是安全方案。



## 总结与延伸

- 你已经完成：规则→数学→实现→测试→攻击/评估。
- 下一步可以：
  - 为实现添加更多字符集与格式化规则
  - 写更强的评分器做自动破译
  - 把多个古典密码组合，体验“组合不等于安全”

> 重要：古典密码主要用于学习思想；真实系统请使用经过标准化与审计的现代密码算法。



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。

